# Prereqs

Step 1 - Navigate to 'Active_CCD_call_result_EH' in user workspace, and run all

Step 2 - Start VPN, navigate to 'ingest00xcuttingstg' in Azure

Step 3 - In ingest00xcuttingstg: af-idempotency -> active, delete relevant state idempotency folder

(Step 3.5 - If unable to delete, go to myaccess Microsoft and self request the 'DLRM Data Ingestion Admin (env:staging)' and wait for it to be granted, then retry deleting)

Step 4 - Run 'Active_Publish_EH_HTML_JSON', found in live -> ACTIVE -> MVP -> Active_Publish_EH_HTML_JSON, specifying relevant state

One of:
- paymentPending
- appealSubmitted
- awaitingRespondentEvidence(a)
- awaitingRespondentEvidence(b)
- caseUnderReview
- reasonsForAppealSubmitted
- listing

Step 5 - Return to 'Active_CCD_call_result_EH' in user workspace and monitor cases coming through

# Setup cells

In [0]:
%pip install azure-monitor-query
%pip install azure-identity
dbutils.library.restartPython()

In [0]:
from azure.identity import ClientSecretCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus
from datetime import datetime, timedelta
from pyspark.sql.functions import input_file_name, regexp_extract, regexp_replace, col
from pyspark.sql import Row
import os
import pandas as pd
import ast
from dataclasses import dataclass
import pandas as pd

#Config
run_user = spark.sql("SELECT current_user()").first()[0]
run_tag = "Testing CCD Payload"
run_by_automation_name = "CCD_Payload_Automation"
test_results_path= "/Workspace/Users/" + run_user + "/Results/CCD_Payload_Tests"
run_start_datetime = datetime.now()

#Create Results Folder
os.makedirs(test_results_path, exist_ok=True)

#Setup TestResult Class
@dataclass
class TestResult:
    test_field: str =""
    status: str =""
    message: str = ""
    test_from_state: str=""
    test_name: str=""  


#Text Used in locating gold data and also text in log
all_states_text = {
"paymentPending": {
    "state_under_test" : "paymentPending",
    "state_under_test_text" : "pendingPayment",    
},
"appealSubmitted": {
    "state_under_test" : "appealSubmitted",
    "state_under_test_text" : "appealSubmitted",
    
},
"awaitingRespondentEvidence(a)": {
    "state_under_test" : "awaitingRespondentEvidence(a)",
    "state_under_test_text" : "awaitingRespondentEvidence"
    
},
"awaitingRespondentEvidence(b)": {
    "state_under_test" : "awaitingRespondentEvidence(b)",
    "state_under_test_text" : "awaitingRespondentEvidence"
    
},

}

#Define What State To Test
testing_state = "awaitingRespondentEvidence(a)"
state_under_test = all_states_text[testing_state]["state_under_test"]
state_under_test_text = all_states_text[testing_state]["state_under_test_text"]


print(f"state_under_test_text = '{state_under_test_text}'")
print(f"state_under_test = '{state_under_test}'")

In [0]:
#######################
#Spark Config
#######################
config_path = "dbfs:/configs/config.json"
config = spark.read.option("multiline", "true").json(config_path)
first_row = config.first()
env = first_row["env"].strip().lower()
lz_key = first_row["lz_key"].strip().lower()
keyvault_name = f"ingest{lz_key}-meta002-{env}"
client_secret = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-SECRET')
tenant_id = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-TENANT-ID')
client_id = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-ID') 


config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
env_name = config.first()["env"].strip().lower()
lz_key = config.first()["lz_key"].strip().lower()
 
print(f"env_code: {lz_key}")  # This won't be redacted
print(f"env_name: {env_name}")  # This won't be redacted
 
KeyVault_name = f"ingest{lz_key}-meta002-{env_name}"
print(f"KeyVault_name: {KeyVault_name}")
 
# Service principal credentials
client_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-ID")
client_secret = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-TENANT-ID")
 
# Storage account names
curated_storage = f"ingest{lz_key}curated{env_name}"
checkpoint_storage = f"ingest{lz_key}xcutting{env_name}"
raw_storage = f"ingest{lz_key}raw{env_name}"
landing_storage = f"ingest{lz_key}landing{env_name}"
external_storage = f"ingest{lz_key}external{env_name}"
 
 
# Spark config for curated storage (Delta table)
spark.conf.set(f"fs.azure.account.auth.type.{curated_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{curated_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{curated_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{curated_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{curated_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{checkpoint_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{checkpoint_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{checkpoint_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{checkpoint_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{checkpoint_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{raw_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{raw_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{raw_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{raw_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{raw_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{landing_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{landing_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{landing_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{landing_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{landing_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

 
# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{external_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{external_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{external_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{external_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{external_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
# Setting variables for use in subsequent cells
bronze_path = f"abfss://bronze@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
silver_path = f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
audit_path = f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/AUDIT/{state_under_test}"
gold_path = f"abfss://gold@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/{state_under_test}"
 
# Print all variables
variables = {
    # "read_hive": read_hive,
    
    "bronze_path": bronze_path,
    "silver_path": silver_path,
    "audit_path": audit_path,
    "gold_path": gold_path,
    "key_vault": KeyVault_name,
    "state_under_test": state_under_test
 
}
 
display(variables)

# Gathering data from the logs

In [0]:
from azure.identity import ClientSecretCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus
from datetime import datetime, timedelta
import pandas as pd
import re

credential = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret
)

client = LogsQueryClient(credential)
workspace_id = "9db45d41-bfe4-49dd-9a3b-5da0ab1a95d0"

query_validate = """
AppTraces
| where TimeGenerated > ago(1d)
| where Message contains "Validate posting payload"
| project TimeGenerated, Message
| order by TimeGenerated desc
| take 1400
"""
#Setup Log trawling params
log_end_time = datetime.utcnow()
log_start_time = log_end_time - timedelta(days=1)


response = client.query_workspace(
    workspace_id,
    query_validate,
    timespan=(log_start_time, log_end_time)
)

if response.status == LogsQueryStatus.SUCCESS:
    for table in response.tables:
        # Convert to DataFrame
        df_validate = pd.DataFrame(
            data=table.rows,
            columns=table.columns
        )
        
else:
    print(f"Query failed: {response.status}")
    df_validate = pd.DataFrame()

if df_validate.empty:
    print('df_validate is empty... try adjusting the time scale.')
else:
    print(f'Found {len(df_validate)} log records')
    # try:
    #     df_validate.display()
    # except Exception as e:
    #     error_message = str(e)
    #     print(f"Error displaying data frame: {error_message[:300]}")

In [0]:
##############################
#Extract/FilterPayload Data From Logs
##############################

log_data_for_state_df = df_validate
# extract the payload and filter for 'xxx'
log_data_for_state_df['payload_validation'] = log_data_for_state_df['Message'].str.split("json = ", n=1).str[1]
log_data_for_state_df['payload_validation'] = log_data_for_state_df['payload_validation'].str.split("'event_token'", n=1).str[0]

# filter rows where ariastate_under_test_text is xxx
log_data_for_state_df = log_data_for_state_df[
    log_data_for_state_df['payload_validation'].str.contains(f"'ariaDesiredState': '{state_under_test_text}'", na=False)
].copy()

# extract CaseNo
log_data_for_state_df['CaseNo'] = log_data_for_state_df['Message'].str.extract(r"'appealReferenceNumber':\s*'([A-Z]+/\d+/\d+)'")

# rename
log_data_for_state_df = log_data_for_state_df.rename(columns={'TimeGenerated': 'payload_validation_time'})

# DISTINCT LOGIC: Keep the latest record for each CaseNo - sort by time descending, then drop duplicates on CaseNo
log_data_for_state_df = log_data_for_state_df.sort_values('payload_validation_time', ascending=False)
log_data_for_state_df = log_data_for_state_df.drop_duplicates(subset=['CaseNo'])

# final selection
log_data_for_state_df = log_data_for_state_df[['CaseNo', 'payload_validation', 'payload_validation_time']]
log_data_for_state_df = spark.createDataFrame(log_data_for_state_df)
print(f"Found : {str(log_data_for_state_df.count())} - Records for state : {state_under_test}")
log_data_for_state_df.display()

# Gathering gold layer data

In [0]:
########################
#Get Gold / Json Data
########################
json_location = dbutils.fs.ls(f"{gold_path}/")[-1]
latest_json_name = json_location.name
latest_json_path = f"{gold_path}/{latest_json_name}JSON/"

# Create a list of JSON paths for just these batches
# path_list = [f"{batch}JSON/" for batch in recent_batches]

# Read just these paths
# raw_df = spark.read.option("multiLine", "true").json(path_list) \
#               .withColumn("raw_filename", input_file_name())

raw_df = spark.read.option("multiLine", "true").json([latest_json_path]) \
              .withColumn("raw_filename", input_file_name())

# Deduplicate
latest_json_df = raw_df.withColumn(
    "appealReferenceNumber", 
    regexp_replace(
        regexp_extract(col("raw_filename"), r"APPEALS_(.*)\.json", 1),
        "_", 
        "/"
    )
).dropDuplicates(["appealReferenceNumber"])

print(f"Loaded Gold Data - Records : {latest_json_df.count()} from gold path : {latest_json_path}")

In [0]:
# matches_df.display()
latest_json_df.display()

# Test

In [0]:
#TEST
from pyspark.sql import Row

# all_log_data = matches_df.select("CaseNo", "payload_validation", "appealReferenceNumber").collect()
all_log_data = log_data_for_state_df.select("CaseNo", "payload_validation").collect()

all_test_results = []
cases_tested = 0

#Loop Gold Data and Check all fields are present in payload to CCD validation api
print(f"Total Number of Cases to Compare : {str(latest_json_df.count())}")
try:
    for case in  latest_json_df.collect():    
        case_no = case.appealReferenceNumber                
        cases_tested += 1
        
        #find log entry that matches case no
        found = False
        for log_entry in all_log_data:
            if log_entry['CaseNo'] == case_no:
                log_data = log_entry['payload_validation']
                found = True
                continue
        if found:
            # Clean up the log string
            clean_str = log_data.strip().rstrip(',')
            if not clean_str.endswith('}'): clean_str += '}'
            
            # Convert log string to dictionary
            try:
                log_json = ast.literal_eval(clean_str).get('data', {})
            except:
                clean_str += '}'
                log_json = ast.literal_eval(clean_str).get('data', {})


            #Compare Fields in Gold with Log Data
            case_dict = case.asDict()
            # for gold_field, gold_val in gold_data.items():
            for gold_field, gold_val in case_dict.items():
                # Skip technical columns from the join
                if gold_field in ['CaseNo', 'payload_validation', 'appealReferenceNumber']:
                    continue

                if gold_field in log_json:
                    #Check if row object for gold, if so convert to dict
                    # if isinstance(gold_val, Row):
                    #     clean_gold_val_dict = rows_to_dicts(gold_val)
                    #     # gold_val = gold_val.asDict()
                    #     log_json_data = log_json[gold_field]
                    #     #Sort Dicts before compare
                    #     sorted_gold_val_dict = dict(sorted(clean_gold_val_dict.items()))
                    #     log_json_data = dict(sorted(log_json_data.items()))
                    # else:
                    log_json_data = log_json[gold_field]

                    # Force to string for a fair comparison
                    s_gold = str(gold_val).strip()
                    s_log = str(log_json_data).strip()
                                        
                    if s_gold != s_log:                
                        all_test_results.append(TestResult(gold_field, "FAIL",f"Failed to Match value for Gold Field : {gold_field} - Expected : {s_gold} -- Actual : {s_log}", state_under_test,case_no))  
                    else:
                        all_test_results.append(TestResult(gold_field, "PASS",f"Passed : Gold Field and Value found in CCD validation payload: {gold_field} / {s_gold}", state_under_test,case_no))           
                else:
                    all_test_results.append(TestResult(gold_field, "FAIL",f"Failed to Find Gold Field : {gold_field} in Validation Payload to CCD", state_under_test,case_no)) 
            
        else:
            all_test_results.append(TestResult(gold_field, "FAIL",f"Failed to Find Gold Case Number : {case_no} - in CCD Validation Data", state_under_test,case_no)) 

except Exception as e:
    error_message = str(e)
    all_test_results.append(TestResult(gold_field, "FAIL",f"Exception Trying to process case : {case_no} - error : {error_message}", state_under_test,"CCD Validation")) 

print(f"Successfully compared {cases_tested} matching cases")
#Get Counts
pass_count = sum(1 for r in all_test_results if r.status.upper() == "PASS")
fail_count = sum(1 for r in all_test_results if r.status.upper() == "FAIL")
total_count = pass_count + fail_count
run_status = "PASS" if fail_count == 0 and pass_count >= 1 else "FAIL"
print(f"OVERALL TEST RESULTS for state : {state_under_test} - Status :  {run_status}  Overall : {str(total_count)} -- Pass: {str(pass_count)}, Fail: {str(fail_count)}")

display(all_test_results)

In [0]:
#Setup Report Filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"{timestamp}_{state_under_test}_CCD_payload_validation_results.csv"
output_path = os.path.join(test_results_path, filename)

#Save to CSV File
results = pd.DataFrame([r.__dict__ for r in all_test_results])
results.to_csv(output_path, index=False)
print(f"File saved to: {output_path}")

display(results)

In [0]:
###############################
#Update Central Results Table
###############################
import uuid
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import *
from datetime import datetime, timedelta

#Define Schema
def run_and_result_table_schemas():

    runs_schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("run_user", StringType(), True),
        StructField("run_start_datetime", TimestampType(), True),
        StructField("run_end_datetime", TimestampType(), True),    
        StructField("run_by_automation_name", StringType(), True),
        StructField("run_tag", StringType(), True),
        StructField("run_status", StringType(), True),
        StructField("state_under_test", StringType(), True), 
        StructField("total_passed", IntegerType(), True), 
        StructField("total_failed", IntegerType(), True), 
        StructField("total_tests", IntegerType(), True), 
    ])

    results_schema = StructType([
        StructField("result_id", StringType(), True),
        StructField("run_id", StringType(), True),  # FK to test_runs
        StructField("test_name", StringType(), True),
        StructField("test_field", StringType(), True),
        StructField("test_from_state", StringType(), True),
        StructField("status", StringType(), True),
        StructField("message", StringType(), True)
    ])
    return runs_schema, results_schema


###############################
#Create Run First (so can get ID to register results against)
###############################
def create_run(run_user, run_start_datetime, run_end_datetime, run_by_automation_name, run_tag, run_status,state_under_test,pass_count, fail_count, total_count):        
    try:
        run_id = str(uuid.uuid4())
        runs_schema, results_schema = run_and_result_table_schemas()
        df = spark.createDataFrame(
            [(run_id, run_user, run_start_datetime, run_end_datetime, run_by_automation_name, run_tag, run_status,state_under_test,pass_count, fail_count, total_count)],runs_schema)
        # df.show()        
        df.write.option("mergeSchema", "true").mode("append").saveAsTable("test_automation_runs")            
        return run_id
    except Exception as e:
        error_message = str(e)  
        print(f"Failed to Create New Run Record. error : {message}")  
        return None
          
###############################
#Create Each Test Result
###############################
def create_results(run_id):
    try:
        runs_schema, results_schema = run_and_result_table_schemas()
        rows = []        
        for result in all_test_results:                               
            try:
                rows.append(
                    (
                    str(uuid.uuid4()),
                    run_id,
                    str(getattr(result, "test_name", "") or ""),
                    str(getattr(result, "test_field", "") or ""),
                    str(getattr(result, "test_from_state", "") or ""),
                    str(getattr(result, "status", "") or ""),
                    str(getattr(result, "message", "") or "")
                    )
                )        
            except:
                print("stop")
        df = spark.createDataFrame(rows, results_schema)
        df.write.option("mergeSchema", "true").mode("append").saveAsTable("test_automation_results")    
    except Exception as e:
        error_message = str(e)   
        print(f"failed to Update Record : {result.test_name} - error : {message}")




In [0]:
#####################
#Update Central results
#####################

#Push Results into Spark Table
print("Starting Pushing Run/Results into Tables")
run_end_datetime = datetime.now()


#Create Run
run_id = create_run(run_user, run_start_datetime, run_end_datetime, run_by_automation_name, run_tag, run_status,state_under_test,pass_count, fail_count, pass_count + fail_count)
print(f"Finsihed creating Run -- Run_id = {str(run_id)}")

#Create Results
if run_id != None:
    create_results(run_id)
    print(f"Finsihed creating Results")
else:
    print("Failed to Create a Run, No results have been submitted to spark tables")